**Vertical profile of NPP for the first hundred meter**

from forecast and hindcast datasets

**Vertical profile from forecast dataset**

In [ ]:
# Import libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import datetime
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import cmocean
import scipy

# To avoid warning messages
import warnings
warnings.filterwarnings('ignore')

In [ ]:
epu_shp = r'\Users\ratna\OneDrive\Documents\NPP\EPUs\EPUs.shp'
gdf = gpd.read_file(epu_shp)

gdf = gdf.to_crs(epsg=4326)

In [ ]:
# Load dataset
foredata = 'Raw Data1/cmems_mod_nwa_bgc-npp_forecast100.nc'
npp_fore = xr.open_dataset(foredata)
npp_fore

In [ ]:
# Initialize variable and depth
npp = npp_fore['nppv']
depth = npp['depth']
print(depth)

# Calculate the mean SST, averaging across latitude, longitude, and time
npp_mean = npp.mean(skipna = True, dim=['latitude', 'longitude', 'time'])
print(npp_mean)

# Plotting vertical profile of NPP
plt.figure(figsize=(6, 10))
plt.plot(npp_mean, depth, label='NPP', color='blue')
plt.xlabel('NPP ($\mathrm{mgC}$ $\mathrm{m^{-3}}$ $\mathrm{day^{-1}}$)')
plt.ylabel('Depth (meter)')
#plt.title('Mean NPP')
plt.grid(False)
plt.gca().invert_yaxis()
plt.gca().xaxis.set_label_position('top')
plt.gca().xaxis.set_ticks_position('top')

# Now we can save a high resolution (300dpi) version of the figure:
#plt.savefig('Forecast profile.png', format = 'png', dpi = 500, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
print(npp_fore)

In [ ]:
npp_fore.isnull()

In [ ]:
npp_fore2 = npp_fore.fillna(0)

In [ ]:
# Depth integration to get surface data which has unit of m2

npp_fore_m2 = npp_fore2.integrate('depth')

npp_fore_m2

In [ ]:
# Initialize variable, depth, and time
npp = npp_fore['nppv']
depth = npp['depth']
time = npp['time']

# Make sure the dimension of 'time' is datetime64, if not, convert time to datetime (skip if it's done)
if not np.issubdtype(time.dtype, np.datetime64):
    time = xr.coding.times.decode_cf_time(npp.time)

# Group by month and calculate the mean across latitude, longitude, and time
npp_monthly = npp.groupby('time.month').mean(skipna = True, dim=['latitude', 'longitude', 'time'])

# Initialize name of the months
month_name = [
    "January", "February", "March", "April", "May", "June", 
    "July", "August", "September", "October", "November", "December"
]

# Plotting vertical profile of NPP
plt.figure(figsize=(8, 12))

# Loop script to create the plot per month
for month in range(12):
    plt.plot(npp_monthly[month], depth, label=month_name[month], alpha=0.7)

plt.xlabel('NPP ($\mathrm{mgC}$ $\mathrm{m^{-3}}$ $\mathrm{day^{-1}}$)')
plt.ylabel('Depth (meter)')
#plt.title('Mean NPP per Month')
plt.gca().invert_yaxis()
plt.gca().xaxis.set_label_position('top')  
plt.gca().xaxis.set_ticks_position('top')
plt.grid(False)
plt.legend(title="Month", loc= 'lower right')

# Show the plot
plt.show()

In [ ]:
# Initialize name of the months
month_name = [
    "January", "February", "March", "April", "May", "June", 
    "July", "August", "September", "October", "November", "December"
]

In [ ]:
#test 
npp_new = npp_fore_m2['nppv']
npp_monthly = npp_new.groupby('time.month').mean(skipna=True)/1000

print(f"max NPP: {npp_monthly.max().values}\nmin NPP: {npp_monthly.min().values}")

In [ ]:
# Define number of rows and columns for the multi-panel plots
nrows = 3
ncols = 4

# Define the figure and each axis for the 3 rows and 4 columns
fig, axs = plt.subplots(nrows = nrows, ncols = ncols,
                        subplot_kw = {'projection': ccrs.PlateCarree()},
                        figsize = (20,15), facecolor='w')

plt.subplots_adjust(bottom=0.15, top=0.96, left=0.04, right=0.99,
                    wspace=0.15, hspace=0.02)

#gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.3, wspace=0.2)

axs = axs.flatten()

# Dummy plot element for the shapefile (used in the legend)
shp_legend = mlines.Line2D([], [], color='red', label='EPU boundaries', linewidth=1)

for i in range(1, 13):
    npp2_plot = npp_monthly[i-1, :, :] # Remember that in Python, the data index starts at 0, but the subplot index start at 1.
    ax = axs[i-1]   # Access the correct subplot
    
    p = ax.pcolormesh(npp_monthly.longitude, npp_monthly.latitude, npp2_plot, vmin=np.min(npp2_plot), vmax=np.max(npp2_plot), cmap = cmocean.cm.algae, zorder=1, alpha=0.8)
    
    ax.set_xlim(-78, -40)
    ax.set_ylim(35, 65)
    ax.set_title(month_name[i-1], fontsize = 13, fontweight = 'bold', color = 'k')
    ax.tick_params(axis='both', labelsize=11)
    ax.set_xticks(ax.get_xticks())
    ax.set_yticks(ax.get_yticks())
    if i % ncols == 1: # Add ylabel for the very left subplots
        ax.set_ylabel('Latitude', fontsize = 11, fontweight = 'bold')
    if i > ncols*(nrows-1): # Add xlabel for the bottom row subplots
        ax.set_xlabel('Longitude', fontsize = 11, fontweight = 'bold')

    # Add the shapefile geometries
    ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(),
                      facecolor='none', edgecolor='red', linewidth=0.3, zorder=2)
    
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=2)
    ax.add_feature(cfeature.BORDERS, facecolor='k', linestyle='-.',  linewidth=1, zorder=2)
    ax.coastlines(resolution='50m', linewidth=1, zorder=3)   # Add coastlines 

# Add a colorbar at the bottom:
cbar_ax = fig.add_axes([1.025, 0.25, 0.018, 0.5])   # 0.25 left, 0.1 bottom, width, height
cbar = plt.colorbar(p, cax=cbar_ax, orientation='vertical', extend = 'max',)  # orientation = vertical or horizontal
cbar.ax.tick_params(labelsize=11)
#cbar.set_label(label='Net Primary Productivity ($\mathrm{mgC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=14)

# Add the legend with the shapefile entry
fig.legend(handles=[shp_legend], loc='lower left', ncol=2, fontsize=12, bbox_to_anchor=(1, 0.1))

# Add title
fig.suptitle('Net Primary Productivity ($\mathrm{gC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=20, y=1.0)

# Now we can save a high resolution (300dpi) version of the figure:
#plt.savefig('Forecast_monthly climatology_in g_300.png', format = 'jpg', dpi = 300, bbox_inches='tight')

plt.show()

**Vertical profile from hindcast dataset**

In [ ]:
# Load dataset
hinddata = 'Raw Data1/cmems_mod_nwa_bgc-npp_hindcast100.nc'
npp_hind = xr.open_dataset(hinddata)
npp_hind['depth'] = npp_hind.depth.round(4)
npp_hind

In [ ]:
hinddata1 = 'Raw Data1/cmems_mod_nwa_bgc-npp_hindcast100_int.nc'
npp_hind1 = xr.open_dataset(hinddata1)
npp_hind1['depth'] = npp_hind1.depth.round(4)
npp_hind1

In [ ]:
npp_hind_merged = xr.concat([npp_hind, npp_hind1], dim='time')
npp_hind_merged

In [ ]:
# Initialize variable and depths
npp1 = npp_hind_merged['nppv']
depths = npp1['depth']
#print(depths)

# Calculate the mean across latitude, longitude, and time
npp1_avg = npp1.mean(skipna = True, dim=['latitude', 'longitude', 'time'])
print(npp1_avg)

# Plotting vertical profile of NPP
plt.figure(figsize=(6, 10))
plt.plot(npp1_avg, depths, label='NPP', color='blue')
plt.xlabel('NPP ($\mathrm{mgC}$ $\mathrm{m^{-3}}$ $\mathrm{day^{-1}}$)')
plt.ylabel('Depth (meter)')
#plt.title('Mean NPP')
plt.gca().invert_yaxis()
plt.gca().xaxis.set_label_position('top')
plt.gca().xaxis.set_ticks_position('top')
plt.grid(False)

# Now we can save a high resolution (300dpi) version of the figure:
#plt.savefig('Hindcast profile.png', format = 'png', dpi = 500, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
npp_hind2 = npp_hind_merged.fillna(0)

In [ ]:
# Depth integration to get surface data which has unit of m2

npp_hind_m2 = npp_hind2.integrate('depth')

npp_hind_m2

In [ ]:
# Initialize variable, depth, and time
npp1 = npp_hind_merged['nppv']
depths = npp_hind_merged['depth']
times = npp_hind_merged['time']

# Make sure the dimension of 'time' is datetime64, if not, convert time to datetime (skip if it's done)
if not np.issubdtype(times.dtype, np.datetime64):
    time = xr.coding.times.decode_cf_time(npp1.time)

# Group by month and calculate the mean across latitude, longitude, and time
npp1_monthly = npp1.groupby('time.month').mean(skipna = True, dim=['latitude', 'longitude', 'time'])

# Initialize name of the months
month_name = [
    "January", "February", "March", "April", "May", "June", 
    "July", "August", "September", "October", "November", "December"
]

# Plotting vertical profile of NPP
plt.figure(figsize=(8, 12))

# Loop script to create the plot per month
for month in range(12):
    plt.plot(npp1_monthly[month], depths, label=month_name[month], alpha=0.7)

plt.xlabel('NPP ($\mathrm{mgC}$ $\mathrm{m^{-3}}$ $\mathrm{day^{-1}}$)')
plt.ylabel('Depth (meter)')
#plt.title('Mean NPP per Month')
plt.gca().invert_yaxis()
plt.gca().xaxis.set_label_position('top')  
plt.gca().xaxis.set_ticks_position('top')
plt.grid(False)
plt.legend(title="Month", loc= 'lower right')

# Show the plot
plt.show()

In [ ]:
#test 
npp1_new = npp_hind_m2['nppv']
npp1_monthly = npp1_new.groupby('time.month').mean(skipna=True)/1000

print(f"max NPP: {npp1_monthly.max().values}\nmin NPP: {npp1_monthly.min().values}")

In [ ]:
# Define number of rows and columns for the multi-panel plots
nrows = 3
ncols = 4

# Define the figure and each axis for the 3 rows and 4 columns
fig, axs = plt.subplots(nrows = nrows, ncols = ncols,
                        subplot_kw = {'projection': ccrs.PlateCarree()},
                        figsize = (20,15), facecolor='w')

plt.subplots_adjust(bottom=0.15, top=0.96, left=0.04, right=0.99,
                    wspace=0.15, hspace=0.02)

#gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.3, wspace=0.2)

axs = axs.flatten()

# Dummy plot element for the shapefile (used in the legend)
shp_legend = mlines.Line2D([], [], color='red', label='EPU boundaries', linewidth=1)

for i in range(1, 13):
    npp3_plot = npp1_monthly[i-1, :, :] # Remember that in Python, the data index starts at 0, but the subplot index start at 1.
    ax = axs[i-1]   # Access the correct subplot
    
    p = ax.pcolormesh(npp1_monthly.longitude, npp1_monthly.latitude, npp3_plot, vmax = np.max(npp3_plot), vmin = np.min(npp3_plot), cmap = cmocean.cm.algae, zorder=1)
    
    ax.set_xlim(-78, -40)
    ax.set_ylim(35, 65)
    ax.set_title(month_name[i-1], fontsize = 13, fontweight = 'bold', color = 'k')
    ax.tick_params(axis='both', labelsize=11)
    ax.set_xticks(ax.get_xticks())
    ax.set_yticks(ax.get_yticks())
    if i % ncols == 1: # Add ylabel for the very left subplots
        ax.set_ylabel('Latitude', fontsize = 11, fontweight = 'bold')
    if i > ncols*(nrows-1): # Add xlabel for the bottom row subplots
        ax.set_xlabel('Longitude', fontsize = 11, fontweight = 'bold')

    # Add the shapefile geometries
    ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(),
                      facecolor='none', edgecolor='red', linewidth=0.3, zorder=2)
    
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=2)
    ax.add_feature(cfeature.BORDERS, facecolor='k', linestyle='-.',  linewidth=1, zorder=2)
    ax.coastlines(resolution='50m', linewidth=1, zorder=2)   # Add coastlines

# Add a colorbar at the bottom:
cbar_ax = fig.add_axes([1.025, 0.25, 0.018, 0.5])
cbar = plt.colorbar(p, cax=cbar_ax, orientation='vertical', extend = 'max',)  # orientation = vertical or horizontal
cbar.ax.tick_params(labelsize=11)
#cbar.set_label(label='Net Primary Productivity ($\mathrm{mgC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=14)

# Add the legend with the shapefile entry
fig.legend(handles=[shp_legend], loc='lower left', ncol=2, fontsize=12, bbox_to_anchor=(1, 0.1))

# Add title
fig.suptitle('Net Primary Productivity ($\mathrm{gC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=20, y=1.0)

# Now we can save a high resolution (300dpi) version of the figure:
#plt.savefig('Hindcast_monthly climatology_in g_300.png', format = 'jpg', dpi = 300, bbox_inches='tight')

plt.show()

In [ ]:
# Choose desired variables
fore = npp_fore_m2['nppv']
hind = npp_hind_m2['nppv']

# Combine more variables
npp_combined = xr.Dataset({
    'FORE': fore,
    'HIND': hind,
})

npp_combined
# Save to new netcdf file
#npp_combined.to_netcdf(r'C:\Users\nsuhita\OneDrive - Marine Institute of Memorial University\Documents\NPP\Temporary output\npp_mod1.nc')